# Match ND-LAr (SPINE) and TMS truth, then view vertices or individual particles

Matching follows `match_spine_to_truth_spill.py` (see `TRUTH_SPILL_MATCHING.md`):
1. **Method 1 (IDs)**: SPINE `orig_id` is not the Truth_Spill vertex ID, so interactions are matched by their
   (track ID, PDG) signature with an order-preserving alignment, separately for normal and throughgoing
   (1e15-flagged) interactions. Cross-check only: reliable for normal interactions, not for throughgoing ones.
2. **Method 2 (position)**: Truth_Spill `TrueVtxX/Y/Z` vs SPINE `ixn.vertex` (cm → mm), within `VERTEX_TOL_MM`.
   **This is the reference.**
3. **Method 3 (particles/tracks)**: within each Method 2 vertex, Truth_Spill particles are matched to SPINE
   particles by track ID (confirmed by PDG and birth position), and TMS reco tracks are linked to them through
   `Truth_Info.RecoTrackPrimaryParticleIndex`.

Two plotting entry points at the end: **by vertex** (`draw_vertices`) and **by particle/track match**
(`draw_particles`).

Run inside the `spine-nb` container (`/data` = `~/Downloads`).

In [ ]:
import numpy as np
import pandas as pd
import uproot
import yaml
import plotly.graph_objects as go
from spine.driver import Driver

In [ ]:
data_dir = "/data/"
spine_file = data_dir + "MiniProdN5p3_NDComplex_FHC.spine.full.sanddrift.0000005.MLRECO_SPINE.hdf5"
tms_file = data_dir + "MiniProdN5p3_NDComplex_FHC.tmsreco.full.sanddrift.v4.0000005.TMSRECO.root"

VERTEX_TOL_MM = 1.0
PARTICLE_TOL_MM = 1.0
THROUGHGOING_FLAG = 10**15  # ID flag on external (throughgoing) vertices, in both SPINE orig_id and Truth_Spill IDs

## Colour palette

In [ ]:
NDLAr_colour = '#e63946'

strong_colours = ['#E69F00', '#0072B2', '#009E73', '#CC79A7', '#D55E00', '#56B4E9', '#F0E442']
light_colours = ['#f0c975', '#70b1d9', '#7ecdb9', '#e0adc9', '#e8a176', '#a2d9f7', '#f7f2a1']

## Load files

In [ ]:
tms_root = uproot.open(tms_file)
tms = tms_root["Truth_Spill"].arrays(
    ["SpillNo", "TrueVtxGlobalID", "TrueVtxX", "TrueVtxY", "TrueVtxZ",
     "VertexGlobalID", "TrackId", "PDG", "IsPrimary", "BirthPosition"], library="np")
spill_to_entry = {int(s): e for e, s in enumerate(tms["SpillNo"])}

# Truth_Info (one entry per TMS slice): true hits tagged by global vertex ID and by the
# Truth_Spill track ID they descend from, plus the truth link of each reco track
tms_info = tms_root["Truth_Info"].arrays(
    ["SpillNo", "TrueHitX", "TrueHitY", "TrueHitZ", "TrueHitVertexId", "TrueHitPrimaryId",
     "RecoTrackN", "RecoTrackPrimaryParticleIndex"], library="np")

# Reco_Tree lines up entry by entry with Truth_Info; TrackHitPos[track][hit] = (x, y, z, t)
tms_reco = tms_root["Reco_Tree"].arrays(["nTracks", "nHits", "TrackHitPos"], library="np")

cfg = f'''
base:
  verbosity: warning
build:
  mode: both
  fragments: false
  particles: true
  interactions: true
  units: cm
io:
  reader:
    file_keys: {spine_file}
    skip_unknown_attrs: True
    name: hdf5
'''
driver = Driver(yaml.safe_load(cfg))
spine_events = [driver.process(i) for i in range(len(driver))]
print(f"{len(tms['SpillNo'])} Truth_Spill entries, {len(spine_events)} SPINE events")

In [ ]:
# Index TMS true hits by (Truth_Spill entry, global vertex ID): xyz (mm) and primary track ID per hit
_hits = {}
for s, x, y, z, vid, pid in zip(tms_info["SpillNo"], tms_info["TrueHitX"], tms_info["TrueHitY"],
                                tms_info["TrueHitZ"], tms_info["TrueHitVertexId"], tms_info["TrueHitPrimaryId"]):
    entry = spill_to_entry[int(s)]
    for g in np.unique(vid):
        sel = vid == g
        _hits.setdefault((entry, int(g)), []).append((np.stack([x[sel], y[sel], z[sel]], 1), pid[sel]))
tms_hits_by_vertex = {key: (np.concatenate([h[0] for h in v]), np.concatenate([h[1] for h in v]))
                      for key, v in _hits.items()}

# (Truth_Spill entry, Truth_Spill particle index) -> list of (Truth_Info entry, reco track index)
reco_tracks_by_particle = {}
for i, (s, n_trk) in enumerate(zip(tms_info["SpillNo"], tms_info["RecoTrackN"])):
    for k in range(n_trk):
        p_idx = int(tms_info["RecoTrackPrimaryParticleIndex"][i][k])
        reco_tracks_by_particle.setdefault((spill_to_entry[int(s)], p_idx), []).append((i, k))


def tms_true_hits(entry, gid, track_id=None):
    """TMS true hits (mm) of a vertex, optionally only those descending from one Truth_Spill track ID."""
    xyz, pid = tms_hits_by_vertex.get((entry, gid), (np.empty((0, 3)), np.empty(0, int)))
    return xyz if track_id is None else xyz[pid == track_id]


def reco_track_hits(info_entry, k):
    """Hit positions (mm) of TMS reco track k in Truth_Info / Reco_Tree entry info_entry."""
    n = int(tms_reco["nHits"][info_entry][k])
    return np.asarray(tms_reco["TrackHitPos"][info_entry][k][:n, :3])

## Method 1: match by (track ID, PDG) signature + order-preserving alignment

Normal and throughgoing interactions are separate ID sequences, so each is aligned on its own.

In [ ]:
def tms_signatures(entry, throughgoing=False):
    """Per Truth_Spill vertex of one kind (sorted by global ID): (global ID, set of (track, pdg))."""
    gids = np.sort(tms["TrueVtxGlobalID"][entry])
    gids = gids[(gids >= THROUGHGOING_FLAG) == throughgoing]
    sig = {int(g): set() for g in gids}
    for g, trk, pdg in zip(tms["VertexGlobalID"][entry], tms["TrackId"][entry], tms["PDG"][entry]):
        if int(g) in sig:
            sig[int(g)].add((int(trk), int(pdg)))
    return [(int(g), sig[int(g)]) for g in gids]


def spine_signatures(data, throughgoing=False):
    """SPINE interactions of one kind, in orig_id order: (ixn, set of (track, pdg))."""
    ixns = sorted((x for x in data["truth_interactions"]
                   if (int(x.orig_id) >= THROUGHGOING_FLAG) == throughgoing),
                  key=lambda x: int(x.orig_id))
    sig = {x.id: set() for x in ixns}
    for p in data["truth_particles"]:
        if p.interaction_id in sig:
            sig[p.interaction_id].add((int(p.track_id), int(p.pdg_code)))
    return [(x, sig[x.id]) for x in ixns]


def align(spine_sig, tms_sig):
    """Order-preserving assignment; returns (total score, {spine position: tms position})."""
    n, m = len(spine_sig), len(tms_sig)
    score = np.zeros((n, m))
    for i, (_, s) in enumerate(spine_sig):
        for j, (_, t) in enumerate(tms_sig):
            score[i, j] = (len(t & s) - 0.5 * len(t - s)) / max(len(t), 1)
    best = np.full((n + 1, m + 1), -1e9)
    best[0, :] = 0
    took = np.zeros((n + 1, m + 1), bool)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            take = best[i - 1, j - 1] + score[i - 1, j - 1]
            best[i, j] = max(best[i, j - 1], take)
            took[i, j] = take >= best[i, j - 1]
    i, j, assign = n, m, {}
    while i > 0 and j > 0:
        if took[i, j]:
            assign[i - 1] = j - 1
            i, j = i - 1, j - 1
        else:
            j -= 1
    return best[n, m], assign


def match_by_ids():
    """{(SPINE event, ixn id): (Truth_Spill entry, global vertex ID)}"""
    n_entries = len(tms["SpillNo"])
    tms_sigs = {tg: [tms_signatures(e, tg) for e in range(n_entries)] for tg in (False, True)}
    out = {}
    for ev, data in enumerate(spine_events):
        s_sig = {tg: spine_signatures(data, tg) for tg in (False, True)}
        results = {tg: [align(s_sig[tg], t_sig) for t_sig in tms_sigs[tg]] for tg in (False, True)}
        entry = int(np.argmax([results[False][e][0] + results[True][e][0] for e in range(n_entries)]))
        for tg in (False, True):
            for i, j in results[tg][entry][1].items():
                out[(ev, int(s_sig[tg][i][0].id))] = (entry, tms_sigs[tg][entry][j][0])
    return out


id_matches = match_by_ids()
print(f"{len(id_matches)} SPINE interactions assigned a vertex by IDs")

## Method 2: match by vertex position (reference)

In [ ]:
def tms_vertex_xyz(entry):
    return np.stack([tms["TrueVtxX"][entry], tms["TrueVtxY"][entry], tms["TrueVtxZ"][entry]], axis=1)


def match_by_position(tol_mm=VERTEX_TOL_MM):
    """{(SPINE event, ixn id): (Truth_Spill entry, global vertex ID, distance mm)}"""
    tms_xyz = [tms_vertex_xyz(e) for e in range(len(tms["SpillNo"]))]
    out = {}
    for ev, data in enumerate(spine_events):
        for ixn in data["truth_interactions"]:
            pos = np.asarray(ixn.vertex) * 10.0  # cm -> mm
            best = None
            for entry, xyz in enumerate(tms_xyz):
                dist = np.linalg.norm(xyz - pos, axis=1)
                j = int(dist.argmin())
                if dist[j] <= tol_mm and (best is None or dist[j] < best[2]):
                    best = (entry, int(tms["TrueVtxGlobalID"][entry][j]), float(dist[j]))
            if best is not None:
                out[(ev, int(ixn.id))] = best
    return out


position_matches = match_by_position()
print(f"{len(position_matches)} SPINE interactions matched to a Truth_Spill vertex within {VERTEX_TOL_MM} mm")

## Cross-check: Method 1 vs Method 2

In [ ]:
agree = {False: 0, True: 0}
total = {False: 0, True: 0}
for key, (entry, gid) in id_matches.items():
    if key not in position_matches:
        continue
    tg = gid >= THROUGHGOING_FLAG
    total[tg] += 1
    agree[tg] += position_matches[key][:2] == (entry, gid)
print(f"ID matches that agree with position matches: {agree[False] + agree[True]}/{total[False] + total[True]} "
      f"(normal {agree[False]}/{total[False]}, throughgoing {agree[True]}/{total[True]})")

## Method 3: particle and track matching within each Method 2 vertex

Status per Truth_Spill particle: `matched` (same track ID, PDG and birth position within `PARTICLE_TOL_MM`),
`rejected` (same track ID, PDG or position disagree), `tms_only` (no SPINE particle with that track ID; mostly
neutrals, or particles that never enter ND-LAr).

In [ ]:
def match_particles(entry, gid, ev, ixn_id, tol_mm=PARTICLE_TOL_MM):
    spine_parts = {int(p.track_id): p for p in spine_events[ev]["truth_particles"] if p.interaction_id == ixn_id}
    out = []
    for idx in np.where(tms["VertexGlobalID"][entry] == gid)[0]:
        trk, pdg = int(tms["TrackId"][entry][idx]), int(tms["PDG"][entry][idx])
        birth = np.asarray(tms["BirthPosition"][entry][idx][:3], dtype=float)
        rec = {"tms_particle_idx": int(idx), "track_id": trk, "tms_pdg": pdg,
               "is_primary": bool(tms["IsPrimary"][entry][idx]),
               "reco_tracks": reco_tracks_by_particle.get((entry, int(idx)), []),
               "spine_particle_id": None, "spine_pdg": None, "birth_dist_mm": None}
        p = spine_parts.get(trk)
        if p is None:
            rec["status"] = "tms_only"
        else:
            dist = float(np.linalg.norm(np.asarray(p.position) * 10.0 - birth))  # cm -> mm
            rec.update(spine_particle_id=int(p.id), spine_pdg=int(p.pdg_code), birth_dist_mm=dist)
            rec["status"] = "matched" if (int(p.pdg_code) == pdg and dist <= tol_mm) else "rejected"
        out.append(rec)
    return out


particle_matches = {key: match_particles(entry, gid, *key) for key, (entry, gid, _) in position_matches.items()}

records = [r for recs in particle_matches.values() for r in recs]
print(f"Truth_Spill particles in matched vertices: {len(records)}",
      {s: sum(r["status"] == s for r in records) for s in ("matched", "rejected", "tms_only")})
n_reco = sum(len(r["reco_tracks"]) for r in records)
print(f"TMS reco tracks from matched vertices: {n_reco}, linked to a SPINE particle: "
      f"{sum(len(r['reco_tracks']) for r in records if r['status'] == 'matched')}")

## Vertex table

In [ ]:
rows = []
for (ev, ixn_id), (entry, gid, dist) in sorted(position_matches.items()):
    ixn = spine_events[ev]["truth_interactions"][ixn_id]
    id_match = id_matches.get((ev, ixn_id))
    recs = particle_matches[(ev, ixn_id)]
    rows.append({
        "SPINE_event": ev,
        "ixn_id": ixn_id,
        "orig_id": int(ixn.orig_id),
        "throughgoing": gid >= THROUGHGOING_FLAG,
        "TS_entry": entry,
        "TS_vtx_global_id": gid,
        "dist_mm": round(dist, 4),
        "id_method_agrees": None if id_match is None else id_match == (entry, gid),
        "n_particles_matched": sum(r["status"] == "matched" for r in recs),
        "n_reco_tracks_linked": sum(len(r["reco_tracks"]) for r in recs if r["status"] == "matched"),
        "n_NDLAr_points": len(ixn.points),
        "n_TMS_hits": len(tms_true_hits(entry, gid)),
    })
match_df = pd.DataFrame(rows)
print("ID method agreement (normal):", match_df[~match_df.throughgoing].id_method_agrees.value_counts().to_dict())
print("ID method agreement (throughgoing):", match_df[match_df.throughgoing].id_method_agrees.value_counts().to_dict())
pd.set_option("display.max_rows", 50)
match_df

In [ ]:
# Vertices with hits in both detectors are the interesting ones for track matching
both = match_df[(match_df.n_NDLAr_points > 0) & (match_df.n_TMS_hits > 0)]
print(f"{len(both)} matched vertices have hits in both ND-LAr and TMS")
both

## Particle table

One row per **matched** particle (Method 3). `n_TMS_hits` counts TMS true hits whose `TrueHitPrimaryId` is this
track ID, so it includes hits from the particle's descendants; `n_NDLAr_points` is the SPINE particle's own points.

In [ ]:
prow = []
for (ev, ixn_id), recs in sorted(particle_matches.items()):
    entry, gid, _ = position_matches[(ev, ixn_id)]
    for r in recs:
        if r["status"] != "matched":
            continue
        p = spine_events[ev]["truth_particles"][r["spine_particle_id"]]
        prow.append({
            "SPINE_event": ev,
            "ixn_id": ixn_id,
            "TS_entry": entry,
            "TS_vtx_global_id": gid,
            "throughgoing": gid >= THROUGHGOING_FLAG,
            "track_id": r["track_id"],
            "pdg": r["tms_pdg"],
            "is_primary": r["is_primary"],
            "spine_particle_id": r["spine_particle_id"],
            "birth_dist_mm": round(r["birth_dist_mm"], 4),
            "n_NDLAr_points": len(p.points),
            "n_TMS_hits": len(tms_true_hits(entry, gid, r["track_id"])),
            "reco_tracks": r["reco_tracks"],
            "n_reco_tracks": len(r["reco_tracks"]),
        })
particle_df = pd.DataFrame(prow)
print(f"{len(particle_df)} matched particles")
particle_df.head(20)

In [ ]:
# Particles seen in both detectors: ND-LAr points and TMS hits, and the ones with a TMS reco track
particles_both = particle_df[(particle_df.n_NDLAr_points > 0) & (particle_df.n_TMS_hits > 0)]
particles_reco = particle_df[particle_df.n_reco_tracks > 0]
print(f"{len(particles_both)} matched particles have hits in both detectors; "
      f"{len(particles_reco)} have a TMS reco track")
particles_reco

## Figure setup and detector boxes

In [ ]:
def plot_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, color, name):
    # One continuous path over every edge; None breaks the line between disconnected edges
    x = [x_min, x_max, x_max, x_min, x_min, x_min, x_max, x_max, x_min, x_min, None, x_max, x_max, None, x_min, x_min, None, x_max, x_max]
    y = [y_min, y_min, y_max, y_max, y_min, y_min, y_min, y_max, y_max, y_min, None, y_min, y_min, None, y_max, y_max, None, y_max, y_max]
    z = [z_min, z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max, z_max, None, z_min, z_max, None, z_min, z_max, None, z_min, z_max]
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(color=color, width=3), name=name, showlegend=True))


def setup_figure(figure, title):
    figure.update_layout(
        width=1600, height=900,
        scene=dict(
            xaxis=dict(title="x [mm]", range=[-4500, 4500]),
            yaxis=dict(title="y [mm]", range=[-4000, 1000]),
            zaxis=dict(title="z [mm]", range=[0, 20000]),
            camera=dict(eye=dict(x=2.0, y=4.0, z=0.0), up=dict(x=0, y=-1, z=0)),  # Y is up
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=4),
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=True,
        legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                    bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='Black', borderwidth=1,
                    itemsizing='constant'),
        title=dict(text=title, x=0.5, xanchor='center'),
    )
    figure.data = []
    # Geometry from TMS_Constants.h (mm)
    plot_box(figure, -3730, 3730, -3702.23, 997.77, 11177.5, 18502.5, "blue", "TMS detector")
    plot_box(figure, -3478.48, 3478.48, -2166.71, 829.282, 4179.24, 9135.88, "black", "ND-LAr detector")


def add_points(figure, xyz, colour, name, size=2):
    if len(xyz):
        figure.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                      marker=dict(size=size, color=colour, symbol='circle'),
                                      name=name, hoverlabel_namelength=-1))


def add_vertex_markers(figure, ev, ixn_id, entry, gid, label):
    v = np.asarray(spine_events[ev]["truth_interactions"][ixn_id].vertex) * 10.0
    figure.add_trace(go.Scatter3d(
        x=[v[0]], y=[v[1]], z=[v[2]], mode="markers",
        marker=dict(size=5, color=NDLAr_colour, symbol='square-open'),
        name=f"{label} SPINE vertex [{v[0]:.1f}, {v[1]:.1f}, {v[2]:.1f}]", hoverlabel_namelength=-1))
    k = int(np.where(tms["TrueVtxGlobalID"][entry] == gid)[0][0])
    tx, ty, tz = tms["TrueVtxX"][entry][k], tms["TrueVtxY"][entry][k], tms["TrueVtxZ"][entry][k]
    figure.add_trace(go.Scatter3d(
        x=[tx], y=[ty], z=[tz], mode="markers",
        marker=dict(size=5, color='black', symbol='cross'),
        name=f"{label} Truth_Spill vertex [{tx:.1f}, {ty:.1f}, {tz:.1f}]", hoverlabel_namelength=-1))

## Plot by vertex

For each chosen `match_df` row: all ND-LAr points of the SPINE interaction, all TMS true hits of the vertex, and
both vertex markers.

In [ ]:
def plot_matched_vertex(figure, row, colour_idx):
    strong = strong_colours[colour_idx % len(strong_colours)]
    light = light_colours[colour_idx % len(light_colours)]
    ev, ixn_id, entry, gid = int(row.SPINE_event), int(row.ixn_id), int(row.TS_entry), int(row.TS_vtx_global_id)
    label = f"(ev {ev} ixn {ixn_id} / vtx {gid})"
    ixn = spine_events[ev]["truth_interactions"][ixn_id]
    add_points(figure, np.asarray(ixn.points) * 10.0, strong, f"{label} ND-LAr hits")  # cm -> mm
    hits = tms_true_hits(entry, gid)
    add_points(figure, hits, light, f"{label} TMS true hits ({len(hits)})")
    add_vertex_markers(figure, ev, ixn_id, entry, gid, label)


def draw_vertices(rows):
    """Plot the matched vertices in the given match_df rows."""
    fig = go.Figure()
    setup_figure(fig, f"Matched vertices (match_df rows): {[int(r) for r in rows]}")
    for c, r in enumerate(rows):
        plot_matched_vertex(fig, match_df.loc[r], c)
    fig.show()

In [ ]:
# Choose match_df rows to plot (default: first few vertices with hits in both detectors)
vertex_rows_to_plot = list(both.index[:3]) if len(both) else [0]
draw_vertices(vertex_rows_to_plot)

## Plot by particle / track match

For each chosen `particle_df` row: the SPINE particle's ND-LAr points, the TMS true hits descending from the same
track ID, every TMS reco track linked to it (hits joined by a line), and the vertex markers.

In [ ]:
def plot_matched_particle(figure, row, colour_idx):
    strong = strong_colours[colour_idx % len(strong_colours)]
    light = light_colours[colour_idx % len(light_colours)]
    ev, ixn_id, entry, gid = int(row.SPINE_event), int(row.ixn_id), int(row.TS_entry), int(row.TS_vtx_global_id)
    label = f"(ev {ev} ixn {ixn_id} track {row.track_id}, PDG {row.pdg})"
    p = spine_events[ev]["truth_particles"][int(row.spine_particle_id)]
    add_points(figure, np.asarray(p.points) * 10.0, strong, f"{label} ND-LAr points")  # cm -> mm
    hits = tms_true_hits(entry, gid, int(row.track_id))
    add_points(figure, hits, light, f"{label} TMS true hits ({len(hits)})")
    for info_entry, k in row.reco_tracks:
        rh = reco_track_hits(info_entry, k)
        if len(rh):
            figure.add_trace(go.Scatter3d(
                x=rh[:, 0], y=rh[:, 1], z=rh[:, 2], mode="lines+markers",
                line=dict(color=strong, width=4), marker=dict(size=3, color=strong, symbol='diamond'),
                name=f"{label} TMS reco track (slice {info_entry}, #{k}, {len(rh)} hits)",
                hoverlabel_namelength=-1))
    add_vertex_markers(figure, ev, ixn_id, entry, gid, f"(ev {ev} ixn {ixn_id})")


def draw_particles(rows):
    """Plot the matched particles in the given particle_df rows."""
    fig = go.Figure()
    setup_figure(fig, f"Matched particles (particle_df rows): {[int(r) for r in rows]}")
    for c, r in enumerate(rows):
        plot_matched_particle(fig, particle_df.loc[r], c)
    fig.show()

In [ ]:
# Choose particle_df rows to plot (default: first few particles with a TMS reco track)
particle_rows_to_plot = list(particles_reco.index[:3]) if len(particles_reco) else list(particle_df.index[:1])
draw_particles(particle_rows_to_plot)